# 13 — Fonte de dados: simulador Databricks na AWS

Referência: [AgentCore workshops](https://github.com/awslabs/agentcore-samples/tree/main/06-workshops). O código e o IaC desta jornada adaptam os conceitos às contas do workshop.

## Pré-requisito: 4 contas AWS distintas

**Esta aula não executa em uma conta única.** O fluxo mostrado envolve 2 contas (platform, transaction-banking), mas o manifesto e o IaC atuais exigem as quatro contas preparadas: plataforma, Retail, Transaction e Lending. Sem elas, reserve a aula para demonstração do instrutor no ambiente completo ou retire-a da seleção de execução.

**Runner CodeBuild do projeto:** a implantação atual exige quatro contas e as integrações Registry, Terraform e RAG, inclusive com `--labs`. A seleção parcial não converte esse runner em uma implantação de conta única. Confira a [matriz de contas e execução](../docs/contas-e-elegibilidade.md).

![Arquitetura AWS da aula 13](../docs/diagramas/svg/13_databricks_e_evidencias.svg)

Leia as fronteiras do diagrama e confronte-as com o IaC e as chamadas abaixo.

Todos os serviços e agentes desta jornada são implantados na AWS.
O notebook é o roteiro e cliente das APIs; ferramentas, inferência,
políticas, fontes e memória executam nos serviços indicados.
Use o ambiente AWS descrito no [guia de implantação](../docs/deploy.md).
Configure os identificadores dos stacks antes de executar. Uma
dependência ausente bloqueia a aula antes das chamadas do exercício.
Antes de usar “Run All”, selecione as aulas compatíveis com as contas
preparadas. O bloco 10–14 exige quatro contas na implantação atual.

### Carregar a configuração e conferir as dependências

A configuração contém perfis/roles e IDs, nunca senhas ou tokens. O STS confirma a conta antes de consultar os stacks; o manifesto exige as dependências desta aula.

In [ ]:
import json
import os
import uuid
from pathlib import Path
from IPython.display import Code, display
import inspect
from santander_demo.aws_workshop.client import AWSWorkshop
from santander_demo.aws_workshop.config import load
from santander_demo.aws_workshop.manifest import require

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
CONFIG = Path(os.environ.get("WORKSHOP_CONFIG", ROOT / "config/aws.workshop.json"))
config = load(CONFIG)
dependencies = require(config, "13")
print({"lab": "13", "dependencies": list(dependencies), "region": config["region"]})

### Conectar ao ambiente implantado

Lemos os outputs exatos do CloudFormation. O cliente não cria infraestrutura e não substitui um serviço indisponível.

In [ ]:
lab = AWSWorkshop(CONFIG, domain="transaction-banking")
print(lab.preflight())

### Autenticar a pessoa no Cognito

O login usa USER_PASSWORD_AUTH para usuários sintéticos do workshop e SECRET_HASH para o cliente confidencial. Em uma aplicação de usuários finais, use authorization code com PKCE. O código abaixo mostra a chamada Cognito e a verificação de assinatura, emissor, audiência e token_use. Só imprimimos claims selecionadas.

In [ ]:
from santander_demo.aws_workshop.identity import login, verify
display(Code(inspect.getsource(login), language="python"))
display(Code(inspect.getsource(verify), language="python"))
token = lab.login("c", "supervisor")
claims = lab.claims(token)
print({key: claims[key] for key in ["iss", "sub", "aud", "token_use"]})

O workspace Databricks foi substituído, a pedido do usuário, por um
**simulador hospedado na AWS**: Runtime MCP + DynamoDB. Ele representa
`current_user()` e grants por pessoa com dados sintéticos. Não executa
SQL Warehouse, token exchange Databricks ou Unity Catalog, e não
comprova esses produtos.

### Verificar a identidade na fonte

A fonte valida o JWT recebido. Não aceita user_id fornecido nos argumentos da ferramenta. Compare o subject retornado com aquele verificado no login.

In [ ]:
proof = await lab.call(token, "especialista___identidade_fonte", {})
assert proof["result"]["verified_subject"] == claims["sub"]
print(proof)

### Comparar grants de B e C na fonte

A ferramenta de consulta é permitida pelo Gateway, mas a fonte mantém uma ACL independente. B pode usar a ferramenta genérica e não tem grant na tabela Riscos.

In [ ]:
result = await lab.call(token, "especialista___consultar_tabela", {"tabela": "riscos"})
assert result["result"]["rows"][0]["cliente_id"] == "C001"
denied = await lab.expect_denial(lab.login("b"), "especialista___consultar_tabela",
                                {"tabela": "riscos"}, expected="SOURCE_DENY")
print({"allowed": result, "source_denied": denied})

### Ler a implementação da ACL independente

GetItem lê a ACL com consistência forte antes de buscar a linha de dados. A identidade vem do JWT verificado.

In [ ]:
from santander_demo.aws_workshop.backend import Source
display(Code(inspect.getsource(Source.read), language="python"))

**Databricks real na AWS:** substitua este target por uma integração com workspace AWS e SQL Warehouse, use credencial delegada suportada pelo provedor e valide `SELECT current_user()` mais queries canário permitidas/negadas. Com Entra ID, o princípio é semelhante, mas federation policy, audience e token exchange exigem configuração própria.

### Evidências e encerramento

Registre IDs de requisição e decisões, sem tokens ou senhas. Os recursos são compartilhados entre aulas: a remoção pertence ao pipeline de limpeza do projeto e deve respeitar essa dependência.